# Estimación de CSS en el Río Magdalena mediante Sentinel-2
## Análisis Exploratorio de Datos
**Tesis de Pregrado en Geología · Universidad del Norte**  
**Francisco Javier Morales Carroll** · fcarroll@uninorte.edu.co  
[GitHub](https://github.com/Franco1303) | [LinkedIn](https://www.linkedin.com/in/francisco-morales-4715092a0)

> Este notebook presenta el análisis exploratorio de datos (EDA) desarrollado como parte de la tesis de pregrado
> enfocada en el desarrollo de un modelo empírico para estimar la concentración de sedimentos en suspensión (SSC)
> en el tramo final del río Magdalena mediante imágenes satelitales Sentinel-2.

---

## 1. Introducción

Este estudio se enfoca en el desarrollo de un modelo empírico para estimar la **concentración de sedimentos
en suspensión (SSC)** en el tramo final del río Magdalena a partir de reflectancia superficial del agua
obtenida de imágenes satelitales **Sentinel-2** del programa Copernicus de la Agencia Espacial Europea (ESA).

Para llevar a cabo este estudio es necesario el uso de mediciones *in situ* de SSC que deben ser unidas con
la reflectancia reportada por imágenes satelitales contemporáneas. El dataset final está compuesto por aquellos
puntos de las campañas de campo para los cuales fue posible obtener reflectancia de Sentinel-2 aplicando
criterios de control de calidad rigurosos.

Los datos de campo fueron tomados con un **perfilador LISST** (*laser in situ scattering and transmissometer*),
que permite obtener perfiles verticales de SSC. Estos fueron tomados cada dos semanas en el período
**Junio 2025 – Marzo 2026**, con una tolerancia de **1 día** entre la medición *in situ* y la imagen.


## 2. Contexto

### Área de estudio

El **río Magdalena** es el ecosistema fluvial con la mayor área y extensión en Colombia, cubriendo
**257,438 km²** (24% del territorio nacional). Su cuenca está caracterizada por alta actividad tectónica,
altas pendientes que exceden los 45° y tiene como principales tributarios el Cauca, Sogamoso, San Jorge
y Cesar (Restrepo et al., 2006).

Representa el **mayor contribuyente de sedimentos al Caribe** con una descarga de **144 × 10⁶ t yr⁻¹**
(Higgins et al., 2016) y es el principal influyente en los cambios morfodinámicos del Caribe Colombiano.

### Estaciones de muestreo

Las estaciones del **kilómetro 5 y 7** fueron descartadas del análisis final por estar fuertemente afectadas
por actividades de dragado. Las estaciones **0, 1 y 3** también introducen este tipo de ruido en menor medida.

### Estación Calamar e Inkora (IDEAM)

La estación hidrológica del IDEAM en Calamar (~100 km de Barranquilla) registra **caudal (Q)** y
**carga sólida total (TSS)**, de los cuales puede derivarse SSC = TSS/Q. Parte del caudal se pierde
hacia el **Canal del Dique**; la estación Inkora monitorea ese corredor fluvial.


## 3. Planteamiento del Problema

El monitoreo de la CSS en ríos de gran caudal como el Magdalena representa un **desafío logístico y
económico** considerable. Los métodos tradicionales requieren campañas de campo intensivas con equipos
especializados como el perfilador LISST.

La teledetección satelital con Sentinel-2 ofrece una alternativa de bajo costo con cobertura sistemática.
Sin embargo, en entornos estuarinos la estimación de CSS es compleja por:
- Interferencia de otros constituyentes ópticos (CDOM, fitoplancton)
- Efectos de marea que desacoplan la señal espectral
- Actividades de dragado que generan variaciones abruptas


## 4. Objetivos

### Objetivo General

> **Estimar** la concentración superficial de sedimento en suspensión (SSC) en el sector fluvial entre
> Calamar y Bocas de Ceniza mediante un **modelo empírico** derivado de variables espectrales satelitales
> e información hidrosedimentológica *in situ*.

### Objetivos Específicos

1. **Caracterizar** la respuesta espectral del agua asociada a diferentes concentraciones de SSC,
   utilizando bandas del visible, NIR y SWIR e índices espectrales derivados.

2. **Calibrar y validar** un modelo empírico de estimación de SSC a partir de variables espectrales
   satelitales, empleando información hidrosedimentológica *in situ*.

3. **Cuantificar** la variabilidad espaciotemporal de la SSC superficial, incluyendo estacionalidad,
   eventos extremos y tendencias.


## 5. Marco Teórico

### Teledetección de sedimentos en suspensión

La estimación de CSS se basa en la relación entre la **reflectancia espectral** del agua y la concentración
de partículas. Los sedimentos **aumentan la reflectancia** en bandas roja y NIR al incrementar la
retrodispersión, mientras que la absorción domina en azul y verde.

### Bandas Sentinel-2 MSI relevantes

| Banda | λ central (nm) | Resolución | Relevancia para SSC |
|-------|---------------|-----------|---------------------|
| Aerosol (B1) | 443 | 60 m | Corrección atmosférica |
| **Blue (B2)** | 492 | **10 m** | Sensible a sedimentos finos |
| **Green (B3)** | 560 | **10 m** | Pico de reflectancia del agua |
| **Red (B4)** | 665 | **10 m** | Clave para NDTI |
| Red Edge 1 (B5) | 704 | 20 m | Sensible a fitoplancton |
| Red Edge 2 (B6) | 740 | 20 m | Complementa B5 |
| Red Edge 3 (B7) | 783 | 20 m | Separa sedimentos/clorofila |
| **NIR (B8)** | 833 | **10 m** | Alta reflectancia en agua turbia |
| Red Edge 4 (B8A) | 865 | 20 m | Alta sensibilidad a SSC |
| SWIR1 (B11) | 1614 | 20 m | Penetra neblina leve |
| SWIR2 (B12) | 2202 | 20 m | Discrimina sedimentos costeros |

### Índices espectrales evaluados

| Índice | Fórmula | Descripción |
|--------|---------|-------------|
| RANS | (Red+NIR)/(Red+NIR+Blue+Green+SWIR1+SWIR2) | Índice normalizado para sedimentos |
| VNES | (Red+RE1+NIR)/(Blue+Green+Red+NIR+SWIR1+SWIR2) | Variante con red edge |
| NDTI | (Red−Green)/(Red+Green) | Índice de turbidez normalizado |
| NIR/RED | NIR/Red | Relación simple NIR/rojo |

### Fórmulas derivadas

**Transporte de sedimentos en suspensión (TSS):**
$$\text{TSS [ton/día]} = \text{SSC [mg/L]} \times Q \text{[m³/s]} \times 0.0864$$

**SSC derivado de TSS y caudal:**
$$\text{SSC [mg/L]} = \frac{\text{TSS [Kt/día]} \times 10^6 / 86400}{Q \text{[m³/s]}} \times \frac{10^6}{10^3}$$

**Corrección por Canal del Dique:**
$$Q_{\text{sinincora}} = Q_{\text{calamar}} - (0.080649 \times Q_{\text{calamar}} - 126.19)$$


## 6. Configuración e Importaciones

In [ ]:
import pathlib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

# Paleta de colores por km
KM_COLORS = {
    0:'#1a6b9a', 1:'#c5e71c', 3:'#1aff00',
    11:'#2eaa6b', 14:'#6a00ed', 17:'#e07b2a',
    18:'#c0392b', 19:'#7b52ab'
}

BANDAS  = ['aerosol','blue','green','red','rojo 1','rojo 2',
           'rojo 3','NIR','rojo 4','SWIR1','SWIR2']
INDICES = ['RANS','VNES','NDTI','NIR/RED']
SSCS    = ['SSC','SSC2','SSC4']
MESES   = ['Ene','Feb','Mar','Abr','May','Jun',
           'Jul','Ago','Sep','Oct','Nov','Dic']

print('✅ Librerías cargadas correctamente')

## 7. Carga de Datos

In [ ]:
# Dataset principal
df = pd.read_csv('puntos_alternativos.csv')
df['reflectance_date'] = pd.to_datetime(df['reflectance_date'])
df['scc_date']         = pd.to_datetime(df['scc_date'])
df['mes']              = df['reflectance_date'].dt.month

print(f'📊 Observaciones: {len(df)}')
print(f'📍 Estaciones (km): {sorted(df["km"].unique())}')
print(f'📅 Período: {df["reflectance_date"].min().date()} → {df["reflectance_date"].max().date()}')
print(f'🌊 Rango SSC: {df["SSC"].min():.1f} – {df["SSC"].max():.1f} mg/L')
df.head(3)

In [ ]:
# Datos hidrológicos
Q_cal = pd.read_csv('Q_MEDIA_D@29037020.data', sep='|')
Q_cal.columns = ['Fecha','Q_calamar']
Q_cal['Fecha'] = pd.to_datetime(Q_cal['Fecha'])

TSS_cal = pd.read_csv('TR_KT_D_QS_D@29037020.data', sep='|')
TSS_cal.columns = ['Fecha','TSS_calamar']
TSS_cal['Fecha'] = pd.to_datetime(TSS_cal['Fecha'])

Q_baq = pd.read_excel('caudal_ganara.xlsx')
Q_baq.columns = ['Fecha','Q_barranquilla']
Q_baq['Fecha'] = pd.to_datetime(Q_baq['Fecha'])

# Merge y variables derivadas
hydro = Q_cal.merge(TSS_cal, on='Fecha', how='inner')
hydro['ssc_derived']  = ((hydro['TSS_calamar']*(1e6/86400)) / hydro['Q_calamar']) * (1e6/1e3)
hydro['Q_sinincora']  = hydro['Q_calamar'] - (0.080649*hydro['Q_calamar'] - 126.19)
hydro['mes']          = hydro['Fecha'].dt.month

print(f'Q Calamar: {Q_cal["Fecha"].min().year}–{Q_cal["Fecha"].max().year} ({len(Q_cal):,} registros)')
print(f'TSS Calamar: {TSS_cal["Fecha"].min().year}–{TSS_cal["Fecha"].max().year} ({len(TSS_cal):,} registros)')
print(f'Q Barranquilla: {len(Q_baq)} semanas')

## 8. Análisis Exploratorio

### 8.1 Estadísticos descriptivos

In [ ]:
cols_s = ['SSC'] + [c for c in BANDAS + INDICES if c in df.columns]
stats  = df[cols_s].describe().T[['mean','std','min','50%','max']].round(4)
stats.columns = ['Media','Desv. Est.','Mín.','Mediana','Máx.']
stats

In [ ]:
# Observaciones por km
df.groupby('km')['SSC'].agg(['count','mean','std','min','max']).round(2)

### 8.2 Distribución de SSC por estación

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Histograma por Km','Boxplot por Km'))

for km in sorted(df['km'].unique()):
    sub = df[df['km']==km]; col = KM_COLORS.get(km,'#1a6b9a')
    fig.add_trace(go.Histogram(x=sub['SSC'], name=f'Km {km}',
        marker_color=col, opacity=0.75, nbinsx=15), row=1, col=1)
    fig.add_trace(go.Box(y=sub['SSC'], name=f'Km {km}',
        marker_color=col, boxmean=True, showlegend=False), row=1, col=2)

fig.update_layout(barmode='overlay', height=420,
    xaxis_title='SSC (mg/L)', legend=dict(orientation='h',y=-0.25))
fig.update_xaxes(showgrid=False)
fig.show()

### 8.3 Series de tiempo

In [ ]:
fig = go.Figure()
for km in sorted(df['km'].unique()):
    sub = df[df['km']==km].sort_values('reflectance_date')
    col = KM_COLORS.get(km,'#1a6b9a')
    fig.add_trace(go.Scatter(x=sub['reflectance_date'], y=sub['SSC'],
        mode='lines+markers', name=f'Km {km}',
        line=dict(color=col,width=2), marker=dict(size=7,color=col)))

fig.update_layout(height=380, title='SSC por estación en el tiempo',
    xaxis_title='Fecha', yaxis_title='SSC (mg/L)',
    xaxis=dict(showgrid=False), legend=dict(orientation='h',y=-0.25))
fig.show()

### 8.4 Firmas espectrales

In [ ]:
WL_NAMES = ['aerosol','blue','green','red','rojo 1','rojo 2',
            'rojo 3','NIR','rojo 4','SWIR1','SWIR2']
WL_REAL  = [443.9,496.6,560,664.5,703.9,740.2,782.5,835.1,864.8,1613.7,2202.4]

sub = df.sort_values('SSC').reset_index(drop=True)
ssc_min, ssc_max = sub['SSC'].min(), sub['SSC'].max()

fig = make_subplots(rows=1, cols=2, column_widths=[0.73,0.27],
    shared_yaxes=True, horizontal_spacing=0.05,
    subplot_titles=['Visible / NIR (400–950 nm)','SWIR'])

for _, row in sub.iterrows():
    t   = (row['SSC']-ssc_min)/(ssc_max-ssc_min+1e-9)
    col = f'rgb(255,{int(165*(1-t))},0)'
    vis = [row[b] for b in WL_NAMES[:9]]
    hover = f"SSC: {row['SSC']:.1f} mg/L | {str(row['reflectance_date'])[:10]} | Km {row['km']}"
    fig.add_trace(go.Scatter(x=WL_REAL[:9], y=vis, mode='lines+markers',
        line=dict(color=col,width=2), marker=dict(size=6,color=col),
        hovertemplate=hover+'<extra></extra>', showlegend=False), row=1, col=1)
    for si,wf in [(9,64),(10,232)]:
        fig.add_trace(go.Scatter(x=[wf], y=[row[WL_NAMES[si]]], mode='markers',
            marker=dict(size=8,color=col), showlegend=False), row=1, col=2)

all_r = df[[b for b in WL_NAMES if b in df.columns]].values.flatten()
y_min = max(0, np.nanmin(all_r)*0.9); y_max = np.nanmax(all_r)*1.08

fig.update_xaxes(title_text='Longitud de onda (nm)',showgrid=False,range=[400,950],row=1,col=1)
fig.update_xaxes(tickvals=[64,114,232,282],ticktext=['1614','1650','2202','2250'],
    showgrid=False,range=[0,320],row=1,col=2)
fig.update_yaxes(title_text='Reflectancia (sr⁻¹)',range=[y_min,y_max],row=1,col=1)
fig.update_layout(height=500, hovermode='closest',
    title='Firmas espectrales coloreadas por SSC (amarillo=bajo, rojo=alto)')
fig.show()

### 8.5 Scatter reflectancia / SSC

In [ ]:
def scatter_ssc(data, x_var, y_var='SSC', transform='log', ajuste='lineal'):
    """Scatter plot con ajuste de regresión lineal o potencial."""
    data = data.dropna(subset=[x_var, y_var])
    x    = data[x_var]; y_raw = data[y_var]
    y    = np.log(y_raw) if transform=='log' else y_raw
    y_lbl= f'ln({y_var})' if transform=='log' else f'{y_var} (mg/L)'

    if ajuste=='lineal':
        ok   = (~np.isnan(x)) & (~np.isnan(y))
        r,p  = pearsonr(x[ok],y[ok])
        m,b  = np.polyfit(x[ok],y[ok],1)
        xl   = np.linspace(x.min(),x.max(),200)
        yl   = m*xl+b
        eq   = f'y = {m:.4f}x + {b:.4f}'
    else:
        ok   = (x>0)&(y_raw>0)&(~np.isnan(x))
        lx   = np.log(x[ok]); ly = np.log(y_raw[ok])
        r,p  = pearsonr(lx,ly)
        b_e,la = np.polyfit(lx,ly,1); a = np.exp(la)
        xl   = np.linspace(x[ok].min(),x[ok].max(),200)
        yl   = np.log(a*(xl**b_e)) if transform=='log' else a*(xl**b_e)
        eq   = f'y = {a:.4f}·x^{b_e:.4f}'

    fig = go.Figure()
    for km in sorted(data['km'].unique()):
        s  = data[data['km']==km]; col = KM_COLORS.get(km,'#1a6b9a')
        ys = np.log(s[y_var]) if transform=='log' else s[y_var]
        fig.add_trace(go.Scatter(x=s[x_var],y=ys,mode='markers',name=f'Km {km}',
            marker=dict(size=9,color=col,line=dict(width=1,color='white'))))
    fig.add_trace(go.Scatter(x=xl,y=yl,mode='lines',name='Regresión',
        line=dict(color='#c0392b',width=2,dash='dash')))

    p_txt = '< 0.0001' if p<0.0001 else f'{p:.4f}'
    r2 = r**2
    print(f'{x_var}: R²={r2:.3f}  p={p_txt}  {eq}')
    fig.update_layout(title=f'<b>{x_var} vs {y_lbl}</b>  |  {eq}  |  R²={r2:.3f}  p={p_txt}',
        xaxis=dict(title=x_var,showgrid=False),
        yaxis=dict(title=y_lbl,gridcolor='#e2e6ea'),
        height=420, legend=dict(orientation='h',y=-0.25))
    return fig

# Ajuste lineal — banda roja
scatter_ssc(df,'red',transform='log',ajuste='lineal').show()

In [ ]:
scatter_ssc(df,'NIR',transform='log',ajuste='lineal').show()

In [ ]:
# Ajuste potencial — comparación
if 'rojo 3' in df.columns:
    scatter_ssc(df,'rojo 3',transform='log',ajuste='potencial').show()

### 8.6 Ranking de correlaciones

In [ ]:
transform = 'log'
y_css  = np.log(df['SSC']) if transform=='log' else df['SSC']
y_lbl  = 'ln(CSS)' if transform=='log' else 'CSS'

results = []
for col in BANDAS + INDICES:
    if col not in df.columns: continue
    x  = df[col]; ok = x.notna() & y_css.notna()
    if ok.sum()<3: continue
    r,p = pearsonr(x[ok],y_css[ok])
    results.append({'variable':col,'r':r,'r_abs':abs(r),'p':p})

res    = pd.DataFrame(results).sort_values('r_abs',ascending=True)
colors = ['#2eaa6b' if r>=0 else '#c0392b' for r in res['r']]
p_lbl  = [f'<0.001' if p<0.001 else f'{p:.3f}' for p in res['p']]

fig = go.Figure(go.Bar(
    x=res['r'], y=res['variable'], orientation='h',
    marker=dict(color=colors),
    text=[f'r={r:.3f}  p={p}' for r,p in zip(res['r'].round(3),p_lbl)],
    textposition='outside'
))
fig.add_vline(x=0,  line=dict(dash='dash',color='#1c2331'))
fig.add_vline(x=0.7, line=dict(color='#2eaa6b',dash='dot',width=1), opacity=0.6)
fig.add_vline(x=-0.7,line=dict(color='#c0392b',dash='dot',width=1), opacity=0.6)
fig.update_layout(height=max(320,len(res)*32+80),
    title='Ranking de correlaciones con SSC',
    xaxis=dict(title=f'Correlación de Pearson con {y_lbl}',range=[-1.2,1.2]),
    yaxis=dict(showgrid=False), showlegend=False)
fig.show()

print('\nTop 5 correlaciones:')
res.sort_values('r_abs',ascending=False).head(5)[['variable','r','p']].round(4)

### 8.7 Matriz de correlación

In [ ]:
transform_c = 'log'
cols_c = [c for c in BANDAS+INDICES+['SSC'] if c in df.columns]
dc     = df[cols_c].copy()
if transform_c=='log': dc['SSC'] = np.log(dc['SSC'])
labels = ['ln(CSS)' if c=='SSC' and transform_c=='log' else c for c in cols_c]

cm = dc.corr()
fig = go.Figure(go.Heatmap(z=cm.values,x=labels,y=labels,
    colorscale='RdBu',zmid=0,zmin=-1,zmax=1,
    text=np.round(cm.values,2),texttemplate='%{text}',textfont={'size':9},
    hovertemplate='X: %{x}<br>Y: %{y}<br>r = %{z:.3f}<extra></extra>'))
fig.update_layout(height=520,xaxis=dict(tickangle=-45),
    title='Matriz de correlación de Pearson')
fig.show()

### 8.8 Mapa de calor espacio-temporal

In [ ]:
df['fecha_str'] = df['reflectance_date'].dt.strftime('%Y-%m-%d')
pivot = (df.groupby(['km','fecha_str'])['SSC']
           .mean().reset_index()
           .pivot(index='km',columns='fecha_str',values='SSC')
           .sort_index(ascending=False))

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns.tolist(),
    y=[f'Km {k}' for k in pivot.index],
    colorscale='YlOrRd', xgap=2, ygap=2,
    colorbar=dict(title='SSC (mg/L)'),
    hovertemplate='Fecha: %{x}<br>%{y}<br>SSC: %{z:.1f} mg/L<extra></extra>'
))
fig.update_layout(height=max(300,len(pivot)*60+100),
    title='SSC promedio por estación y fecha de imagen',
    xaxis=dict(title='Fecha de imagen',tickangle=-45,showgrid=False),
    yaxis=dict(showgrid=False))
fig.show()

### 8.9 Climatograma de SSC

In [ ]:
fig = go.Figure()

for m in range(1,13):
    sub = df[df['mes']==m]
    if sub.empty: continue
    fig.add_trace(go.Box(y=sub['SSC'],x=[MESES[m-1]]*len(sub),
        showlegend=False,marker=dict(color='#1a6b9a',opacity=0.35,size=4),
        line=dict(color='#1a6b9a'),boxmean=True,hoverinfo='skip'))

for km in sorted(df['km'].unique()):
    sub = df[df['km']==km]; col = KM_COLORS.get(km,'#1a6b9a')
    fig.add_trace(go.Scatter(x=[MESES[m-1] for m in sub['mes']],y=sub['SSC'],
        mode='markers',name=f'Km {km}',
        marker=dict(size=8,color=col,line=dict(width=1,color='white'),opacity=0.9)))

means = df.groupby('mes')['SSC'].mean().reindex(range(1,13))
fig.add_trace(go.Scatter(
    x=[MESES[m-1] for m in means.index if not pd.isna(means[m])],
    y=[v for v in means.values if not pd.isna(v)],
    mode='lines+markers',name='Media mensual',
    line=dict(color='#1c2331',width=2.5,dash='dash'),
    marker=dict(size=8,color='#1c2331')))

fig.update_layout(boxmode='overlay',height=450,
    title='Distribución mensual de SSC — período de estudio',
    xaxis=dict(title='Mes',categoryorder='array',categoryarray=MESES,showgrid=False),
    yaxis=dict(title='SSC (mg/L)',gridcolor='#e2e6ea'),
    legend=dict(orientation='h',y=-0.25))
fig.show()

## 9. Hidrología — Calamar y Barranquilla

### 9.1 Series de tiempo

In [ ]:
h_r  = hydro[hydro['Fecha'].dt.year >= 2022]
Qb_r = Q_baq[Q_baq['Fecha'].dt.year  >= 2022]

fig = make_subplots(rows=3,cols=1,shared_xaxes=True,vertical_spacing=0.08,
    subplot_titles=['Caudal (m³/s)','TSS Calamar (Kt/día)','SSC Derivado (mg/L)'])

fig.add_trace(go.Scatter(x=h_r['Fecha'],y=h_r['Q_calamar'],mode='lines',
    name='Q Calamar',line=dict(color='#1a6b9a',width=1.5)),row=1,col=1)
fig.add_trace(go.Scatter(x=h_r['Fecha'],y=h_r['Q_sinincora'],mode='lines',
    name='Q Cal − Q Incora',line=dict(color='red',width=2)),row=1,col=1)

if len(Qb_r)>0:
    fig.add_trace(go.Scatter(x=Qb_r['Fecha'],y=Qb_r['Q_barranquilla'],
        mode='markers+lines',name='Q Barranquilla',
        line=dict(color='#e07b2a',width=2),marker=dict(size=7)),row=1,col=1)

fig.add_trace(go.Scatter(x=h_r['Fecha'],y=h_r['TSS_calamar'],mode='lines',
    name='TSS Calamar',line=dict(color='#c0392b',width=1.5)),row=2,col=1)
fig.add_trace(go.Scatter(x=h_r['Fecha'],y=h_r['ssc_derived'],mode='lines',
    name='SSC Derivado',line=dict(color='#4bb929',width=1.5)),row=3,col=1)

fig.update_layout(height=620,legend=dict(orientation='h',y=-0.07))
fig.update_xaxes(showgrid=False)
fig.show()

### 9.2 Estacionalidad

In [ ]:
fig = make_subplots(rows=1,cols=2,
    subplot_titles=['Caudal Calamar (m³/s)','TSS Calamar (Kt/día)'])

for m in range(1,13):
    qv = hydro[hydro['mes']==m]['Q_calamar']
    tv = hydro[hydro['mes']==m]['TSS_calamar']
    if len(qv)>0: fig.add_trace(go.Box(y=qv,name=MESES[m-1],showlegend=False,
        marker_color='#1a6b9a'),row=1,col=1)
    if len(tv)>0: fig.add_trace(go.Box(y=tv,name=MESES[m-1],showlegend=False,
        marker_color='#c0392b'),row=1,col=2)

fig.update_layout(height=440)
fig.update_xaxes(showgrid=False)
fig.show()

### 9.3 Q vs TSS Calamar

In [ ]:
mg   = hydro.dropna(subset=['Q_calamar','TSS_calamar'])
r,p  = pearsonr(mg['Q_calamar'],mg['TSS_calamar'])
m_c,b_c = np.polyfit(mg['Q_calamar'],mg['TSS_calamar'],1)
xl   = np.linspace(mg['Q_calamar'].min(),mg['Q_calamar'].max(),300)
p_t  = '< 0.0001' if p<0.0001 else f'{p:.4f}'

fig = go.Figure()
fig.add_trace(go.Scatter(x=mg['Q_calamar'],y=mg['TSS_calamar'],mode='markers',
    marker=dict(size=4,color='#1a6b9a',opacity=0.45)))
fig.add_trace(go.Scatter(x=xl,y=m_c*xl+b_c,mode='lines',
    line=dict(color='#c0392b',width=2,dash='dash'),name='Regresión'))

fig.update_layout(height=460,
    title=f'Q vs TSS Calamar | y={m_c:.4f}x+{b_c:.2f} | R²={r**2:.3f}  p={p_t}  n={len(mg)}',
    xaxis=dict(title='Q Calamar (m³/s)',showgrid=False),
    yaxis=dict(title='TSS Calamar (Kt/día)',gridcolor='#e2e6ea'))
fig.show()

## 10. Conclusiones

---

**01 — Dataset final de calibración**  
El proceso de control de calidad consolidó entre **27 y 39 observaciones** con R² entre **0.58 y 0.73**,
ubicadas en los kilómetros 0, 1, 3, 14, 17, 18 y 19 del tramo estuarino.

**02 — Bandas más informativas**  
Las bandas **Red, NIR y rojo 3** mostraron las correlaciones más altas con SSC, consistente con la física
de retrodispersión de sedimentos en agua turbia.

**03 — Perspectivas de modelado**  
Con ~30 puntos el dataset es apto para **regresión potencial log-log** y **regresión múltiple** con 2–3
variables, validadas con LOOCV. De ampliarse podría aplicarse Random Forest.

**04 — Limitaciones**  
La influencia de dragados (km 1–11) y los efectos de marea en la zona estuarina representan fuentes
de incertidumbre importantes que limitan tanto el problema como su solución.

---

## Referencias

- Qiu, Z. et al. (2024). Four-decades of sediment transport using Landsat. *RSE*, 306.
- Qiu, Z. et al. (2024). Improving SSC observations from Landsat to Sentinel-2. *IJAEOG*, 134.
- Restrepo, J.D. et al. (2006). Fluvial fluxes into the Caribbean Sea. *GPC*, 50(1–2), 33–49.
- Yepez, S. et al. (2018). SSC retrieval using Landsat-8 in the Orinoco. *CR-Géoscience*, 350, 20–30.
- Gray, J.R. & Simões, F.J.M. (2008). Estimating Sediment Discharge. ASCE.
- Vanoni, V.A. (1975). *Sedimentation Engineering*. ASCE Manual 54.
